In [16]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score


In [17]:
DATA_PATH = "raw_wholesale_customers.csv"
OUT_PATH_DATA = "segmented_wholesale_customers.csv"
RONDOM_STATE = 42
FEATURES = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]
K = 3

In [18]:
# 1) Load dataset

df = pd.read_csv(DATA_PATH)
print("\n=== INITIAL SNAPSHOT ===")
print(df.head())



=== INITIAL SNAPSHOT ===
   Channel  Region  Fresh  Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0        2       3  12669  9656     7561     214              2674        1338
1        2       3   7057  9810     9568    1762              3293        1776
2        2       3   6353  8808     7684    2405              3516        7844
3        1       3  13265  1196     4221    6404               507        1788
4        2       3  22615  5410     7198    3915              1777        5185


In [19]:
# 2) Select features + IQR cap
X = df[FEATURES].copy()


In [20]:
def quant_jarid (sereis, k=1.5):
    Q1 , Q3 = sereis.quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower = Q1 - k * IQR
    upper = Q3 + k * IQR

    return lower, upper

lower_fresh , upper_fresh = quant_jarid(X["Fresh"])
lower_milk , upper_milk = quant_jarid(X["Milk"])
lower_grocery , upper_grocery = quant_jarid(X["Grocery"])
lower_frozen , upper_frozen = quant_jarid(X["Frozen"])
lower_detergents , upper_detergents = quant_jarid(X["Detergents_Paper"])
lower_delicassen , upper_delicassen = quant_jarid(X["Delicassen"])

In [21]:
# cipping
X["Fresh"] = X["Fresh"].clip(lower = lower_fresh , upper = upper_fresh)
X["Milk"] = X["Milk"].clip(lower = lower_milk , upper = upper_milk)
X["Grocery"] = X["Grocery"].clip(lower = lower_grocery , upper = upper_grocery)
X["Frozen"] = X["Frozen"].clip(lower = lower_frozen , upper = upper_frozen)
X["Detergents_Paper"] = X["Detergents_Paper"].clip(lower = lower_detergents , upper = upper_detergents)
X["Delicassen"] = X["Delicassen"].clip(lower = lower_delicassen , upper = upper_delicassen)


# ku celi data aan clip garaynay original data dii 

df[FEATURES] = X

In [22]:
print(f" \n ======== number ka ugu hooseeya ee uu gaadhi karo fresh waa ====== \n === {X["Fresh"].max()} ===")
print("\n=== FEATURES HEAD (after IQR cap) ===")
print(X.head())


 
 ======== number ka ugu hooseeya ee uu gaadhi karo fresh waa ====== 
 === 37642.75 ===

=== FEATURES HEAD (after IQR cap) ===
     Fresh    Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0  12669.0  9656.0   7561.0   214.0            2674.0     1338.00
1   7057.0  9810.0   9568.0  1762.0            3293.0     1776.00
2   6353.0  8808.0   7684.0  2405.0            3516.0     3938.25
3  13265.0  1196.0   4221.0  6404.0             507.0     1788.00
4  22615.0  5410.0   7198.0  3915.0            1777.0     3938.25


In [23]:

# 3) Scale features

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("\nScaled shape:", X_scaled.shape)


Scaled shape: (440, 6)


In [24]:
# 4) Elbow method (print SSE)

print("\n=== ELBOW METHOD (SSE per k) ===")
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init="auto", random_state=RONDOM_STATE)
    km.fit(X_scaled)
    print(f"k={k} → SSE={km.inertia_:.2f}")



=== ELBOW METHOD (SSE per k) ===
k=1 → SSE=2640.00
k=2 → SSE=1728.19
k=3 → SSE=1363.45
k=4 → SSE=1202.41
k=5 → SSE=1070.15
k=6 → SSE=964.76
k=7 → SSE=921.14
k=8 → SSE=776.63
k=9 → SSE=726.88
k=10 → SSE=707.41


In [25]:
# 5) Fit K-Means with chosen k
kmeans = KMeans(n_clusters = K , n_init = "auto", random_state = RONDOM_STATE)
output  = kmeans.fit_predict(X_scaled)

df["Cluster"] = output.astype(int)
print(df.head())

   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \
0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   
1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   
2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   
3        1       3  13265.0  1196.0   4221.0  6404.0             507.0   
4        2       3  22615.0  5410.0   7198.0  3915.0            1777.0   

   Delicassen  Cluster  
0     1338.00        1  
1     1776.00        0  
2     3938.25        2  
3     1788.00        2  
4     3938.25        2  


In [26]:
# 6) Evaluate clustering
# --------------------------------
sil = silhouette_score(X_scaled, output)
dbi = davies_bouldin_score(X_scaled, output)

centers_scaled = kmeans.cluster_centers_
centers_original = scaler.inverse_transform(centers_scaled)

centers_df = pd.DataFrame(centers_original, columns=FEATURES)
centers_df.index.name = "Cluster"

print("\n=== CLUSTER CENTERS (Original Units) ===")
print(centers_df.round(2))
print("\n=== METRICS ===")
print(f"Silhouette Score : {sil:.3f} (closer to +1 is better)")
print(f"Davies–Bouldin   : {dbi:.3f} (lower is better)")


=== CLUSTER CENTERS (Original Units) ===
            Fresh      Milk   Grocery   Frozen  Detergents_Paper  Delicassen
Cluster                                                                     
0         5869.23  10142.25  16653.32  1355.63           7022.05     1482.60
1         9418.37   2685.47   3583.98  2052.33            942.49      750.71
2        22881.83   5870.35   6773.44  5057.43           1209.38     2452.04

=== METRICS ===
Silhouette Score : 0.340 (closer to +1 is better)
Davies–Bouldin   : 1.297 (lower is better)


In [27]:
# 7) Research & train second clustering algorithm
# Agglomerative / Hierarchical Clustering
agg =  AgglomerativeClustering(n_clusters = K, linkage= "ward")
agg_output = agg.fit_predict(X_scaled)

df["Agg_Cluster"] = agg_output.astype(int)

print("\n=== SECOND ALGORITHM: AGGLOMERATIVE CLUSTERING ===")
print("Number of clusters:", K)
print(df[["Cluster", "Agg_Cluster"]].head())



=== SECOND ALGORITHM: AGGLOMERATIVE CLUSTERING ===
Number of clusters: 3
   Cluster  Agg_Cluster
0        1            2
1        0            2
2        2            0
3        2            0
4        2            0


In [28]:
# 8) Compare K-Means and Agglomerative Clustering
kmeans_sil = silhouette_score(X_scaled, output)
agg_sil = silhouette_score(X_scaled, agg_output)

print("\n=== METHOD COMPARISON ===")
print(f"K-Means Silhouette Score        : {kmeans_sil:.3f}")
print(f"Agglomerative Silhouette Score  : {agg_sil:.3f}")

if kmeans_sil > agg_sil:
    print("K-Means produced better-separated clusters based on Silhouette Score.")
elif agg_sil > kmeans_sil:
    print("Agglomerative Clustering produced better-separated clusters based on Silhouette Score.")
else:
    print("Both methods produced the same Silhouette Score.")



=== METHOD COMPARISON ===
K-Means Silhouette Score        : 0.340
Agglomerative Silhouette Score  : 0.284
K-Means produced better-separated clusters based on Silhouette Score.


In [29]:
# Sanity checks 

sample_idx = [5, 12, 9]
sanity = df.loc[sample_idx, ["Channel", "Region"] + FEATURES + ["Cluster","Agg_Cluster" ]]
print("\n=== SANITY CHECK (3 Clients) ===")
print(sanity)


=== SANITY CHECK (3 Clients) ===
    Channel  Region    Fresh     Milk  Grocery  Frozen  Detergents_Paper  \
5         2       3   9413.0   8259.0   5126.0   666.0            1795.0   
12        2       3  31714.0  12319.0  11757.0   287.0            3881.0   
9         2       3   6006.0  11093.0  18881.0  1159.0            7425.0   

    Delicassen  Cluster  Agg_Cluster  
5       1451.0        1            2  
12      2931.0        2            0  
9       2098.0        0            1  


In [30]:
 # 10 Save Output Data
df.to_csv(OUT_PATH_DATA, index=False)
print(f"\nSaved clustered dataset → {OUT_PATH_DATA}")



Saved clustered dataset → segmented_wholesale_customers.csv
